1. 中间文件

In [2]:
import pandas as pd

# 读取原始数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/上市公司人工智能专利.csv')

# 提取需要的列
cols = ['stkcd', '股票简称', 'year', '人工智能专利申请数量', '人工智能专利获得数量']
df_simple = df[cols]

# 保存为新CSV文件
df_simple.to_csv('专利数据简化.csv', index=False, encoding='utf-8-sig')

In [3]:
import pandas as pd

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利数据简化.csv')

# 找到每个stkcd最新年份的证券简称
latest_names = df.sort_values('year').groupby('stkcd').last()['股票简称']

# 用最新证券简称替换所有年份
df['股票简称'] = df['stkcd'].map(latest_names)

# 保存结果
df.to_csv('专利数据简化_统一简称.csv', index=False)

In [4]:
import pandas as pd

# 1. 读取企业规模对照表
scale_df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利_企业规模.csv') 

scale_dict = dict(zip(scale_df['股票简称'], scale_df['企业规模']))

# 2. 读取专利数据
patent_df = pd.read_csv('专利数据简化_统一简称.csv', encoding='utf-8')

# 3. 添加企业规模列
patent_df['企业规模'] = patent_df['股票简称'].map(scale_dict)

# 4. 保存新文件
patent_df.to_csv('专利数据最终文件.csv', index=False, encoding='utf-8-sig')

2. 可视化

In [12]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利数据最终文件.csv')

# 只保留三类企业
df = df[df['企业规模'].isin(['大型企业', '中型企业', '小型企业'])]

# 按年份和企业规模分组，计算平均值
pivot = df.pivot_table(index='year', columns='企业规模', values='人工智能专利申请数量', aggfunc='mean').loc[2016:2023]

# 按企业规模顺序排列
pivot = pivot[['大型企业', '中型企业', '小型企业']]

# 绘制堆积图
# ...existing code...
plt.figure(figsize=(10, 6))
plt.stackplot(
    pivot.index,
    [pivot[col] for col in pivot.columns],
    labels=pivot.columns,
    colors=["#ABC9DB", "#6A93C4", "#4E76CB"],
    alpha=0.85
)
plt.title('2016-2023年不同规模企业平均人工智能专利申请数量（堆积图）')
plt.xlabel('年份')
plt.ylabel('专利申请数量')
plt.legend(title='企业规模')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.xticks(range(2016, 2024))
plt.xlim(2016, 2023)  # 严格限定x轴范围
plt.tight_layout()
plt.show()

<Figure size 720x432 with 1 Axes>

专利转化率可视化

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/专利转化率.csv')

# 只保留四类企业
df = df[df['企业规模'].isin(['大型企业', '中型企业', '小型企业', '微型企业'])]

# 按企业规模分组
pivot = df.pivot(index='年份', columns='企业规模', values='专利转化率')

plt.figure(figsize=(10,6))
ax = plt.gca()

# 先设置x、y轴范围，确保渐变覆盖完整区域
x_min, x_max = min(pivot.index), max(pivot.index)
y_min, y_max = np.nanmin(pivot.values), np.nanmax(pivot.values)
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# 添加浅紫色渐变背景
n = 200  
for i in range(n):
    alpha = 0.15 * (1 - i / n)
    color = (180/255, 160/255, 255/255, alpha)  # 浅紫色RGBA
    y_start = y_min + (y_max - y_min) * i / n
    y_end = y_min + (y_max - y_min) * (i + 1) / n
    plt.axhspan(y_start, y_end, xmin=0, xmax=1, color=color, zorder=0)

plt.plot(pivot.index, pivot['大型企业'], marker='o', label='大型企业', linewidth=2.5, color="#4F77CE")
plt.plot(pivot.index, pivot['中型企业'], marker='s', label='中型企业', linewidth=2.5, color="#F09935")
plt.plot(pivot.index, pivot['小型企业'], marker='^', label='小型企业', linewidth=2.5, color="#93CBA4")  # 浅绿色
plt.plot(pivot.index, pivot['微型企业'], marker='d', label='微型企业', linewidth=2.5, color="#F7A1B0")  # 粉色

# 标注数值
for scale, color in zip(['大型企业', '中型企业', '小型企业', '微型企业'],
                        ["#4F77CE", "#F09935", "#93CBA4", "#F7A1B0"]):
    for x, y in zip(pivot.index, pivot[scale]):
        plt.text(x, y, f'{y:.1f}', ha='center', va='bottom', fontsize=10, color=color)

#plt.title('2016-2023年不同规模企业专利转化率')
plt.xlabel('年份')
plt.ylabel('专利转化率（%）')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.xticks(pivot.index)
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 720x432 with 1 Axes>

单位费用产出专利数

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib import font_manager
# 设置支持中文的字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  
plt.rcParams['axes.unicode_minus'] = False 

# 读取数据
df = pd.read_csv('/home/mw/input/data_total74037403/行业角度数据集/全国企业创新结果.csv', encoding='utf-8')

df = df[df['企业规模'].isin(['大型企业', '中型企业', '小型企业', '微型企业'])]

# 透视表，行为企业规模，列为年份
pivot = df.pivot(index='企业规模', columns='年份', values='单位创新费用产出的专利数')

plt.figure(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.5, cbar_kws={'label': '专利数（件/亿元）'})

plt.title('2016-2023年不同规模企业单位创新费用产出的专利数（热力图）')
plt.xlabel('年份')
plt.ylabel('企业规模')
plt.tight_layout()
plt.show()

<Figure size 720x288 with 2 Axes>